<a href="https://colab.research.google.com/github/ghn9zh/DS3001-programming/blob/main/assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Data Wrangling
### `! git clone https://github.com/ds4e/wrangling`
### Do Q1 and Q2, and either Q3 or Q4, for a total of 3 questions.

In [ ]:
! git clone https://github.com/ds4e/wrangling

Cloning into 'wrangling'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 63 (delta 2), reused 1 (delta 0), pack-reused 56 (from 1)
Receiving objects: 100% (63/63), 13.96 MiB | 8.25 MiB/s, done.
Resolving deltas: 100% (12/12), done.


**Q1.** This question provides some practice cleaning variables which have common problems.
1. Numeric variable: For `./data/airbnb_hw.csv`, clean the `Price` variable as well as you can, and explain the choices you make. How many missing values do you end up with? (Hint: What happens to the formatting when a price goes over 999 dollars, say from 675 to 1,112?)
2. Categorical variable: For the Minnesota police use of for data, `./data/mn_police_use_of_force.csv`, clean the `subject_injury` variable, handling the NA's; this gives a value `Yes` when a person was injured by police, and `No` when no injury occurred. What proportion of the values are missing? Is this a concern? Cross-tabulate your cleaned `subject_injury` variable with the `force_type` variable. Are there any patterns regarding when the data are missing?
3. Dummy variable: For the pretrial data covered in the lecture, clean the `WhetherDefendantWasReleasedPretrial` variable as well as you can, and, in particular, replace missing values with `np.nan`.
4. Missing values, not at random: For the pretrial data covered in the lecture, clean the `ImposedSentenceAllChargeInContactEvent` variable as well as you can, and explain the choices you make. (Hint: Look at the `SentenceTypeAllChargesAtConvictionInContactEvent` variable.)

In [ ]:
import pandas as pd

#Question 1.1
file_path = '/content/wrangling/assignment/data/airbnb_hw.csv'
df = pd.read_csv(file_path)
df.head()

print(df['Price'].dtype)
df['Price'] = df['Price'].str.replace(',', '').astype(int)
print(df['Price'].dtype)
#When the price went above 1000 it turned the integer of price into a
#string because of the comma. I removed the comma and turned all thr values
#into integers.
missing_values = df['Price'].isna().sum()
print(missing_values)
#I end with with 0 missing values after cleaning the column.



object
int64
0


In [ ]:
import pandas as pd

#Question 1.2
file_path = '/content/wrangling/assignment/data/mn_police_use_of_force.csv'
df = pd.read_csv(file_path)
df.head()

df['subject_injury_missing'] = df['subject_injury'].isna() #new column to see if subject_injury was missing
#proportion of missing values
missing_count = df['subject_injury_missing'].sum()
total_count = len(df)
missing_proportion = missing_count / total_count

print(f"Total missing values in 'subject_injury': {missing_count}")
print(f"Proportion of missing values: {missing_proportion}")
#there is around 76% of subject injury values missing which is a big concern
#this could be either they were unknown/unreported or purposefully ommitted which is unethical

cross_table = df.pivot_table(index='force_type', columns='subject_injury_missing', aggfunc='size', fill_value=0)
print(cross_table)
#bodily force and chemical irritant are the most common and least documented for injury which shows there are
#probably less strict rules on documenting these common forces. Firearm injury was reported everytime which
#is probably due to strict regulations where someone is injured from a fire arm is less common and more serious to report.

Total missing values in 'subject_injury': 9848
Proportion of missing values: 0.7619342359767892
subject_injury_missing       False  True 
force_type                               
Baton                            2      2
Bodily Force                  2379   7051
Chemical Irritant              172   1421
Firearm                          2      0
Gun Point Display               77     27
Improvised Weapon               74     74
Less Lethal                      0     87
Less Lethal Projectile           3      0
Maximal Restraint Technique      0    170
Police K9 Bite                  46     31
Taser                          322    985


In [55]:
import pandas as pd
import numpy as np

#Question 1.3
url = 'http://www.vcsc.virginia.gov/pretrialdataproject/October%202017%20Cohort_Virginia%20Pretrial%20Data%20Project_Deidentified%20FINAL%20Update_10272021.csv'
df = pd.read_csv(url,low_memory=False)
df.head()

df = df.rename(columns={'WhetherDefendantWasReleasedPretrial': 'released'})

df['released'] = df['released'].replace(
    ['', ' ', 'NULL', 'NA', 'NaN', 0, 9, 1, None], np.nan
)
unique_values = df['released'].unique()
print(unique_values)


In [ ]:
import pandas as pd

url = 'http://www.vcsc.virginia.gov/pretrialdataproject/October%202017%20Cohort_Virginia%20Pretrial%20Data%20Project_Deidentified%20FINAL%20Update_10272021.csv'
df = pd.read_csv(url, low_memory=False)

df.columns = df.columns.str.strip()

# Convert columns to numeric
df['SentenceTypeAllChargesAtConvictionInContactEvent'] = pd.to_numeric(df['SentenceTypeAllChargesAtConvictionInContactEvent'], errors='coerce')
df['ImposedSentenceAllChargeInContactEvent'] = pd.to_numeric(df['ImposedSentenceAllChargeInContactEvent'], errors='coerce')

missing_before = df['ImposedSentenceAllChargeInContactEvent'].isna().sum()

# Replace missing values with 0
df['ImposedSentenceAllChargeInContactEvent'].fillna(0, inplace=True)

# missing values after cleaning
missing_after = df['ImposedSentenceAllChargeInContactEvent'].isna().sum()

# Print results
print(f"Missing values before cleaning: {missing_before}")
print(f"Missing values after cleaning: {missing_after}")






Missing values before cleaning: 9053
Missing values after cleaning: 0


<ipython-input-41-2fe7d8d2d662>:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['ImposedSentenceAllChargeInContactEvent'].fillna(0, inplace=True)


**Q2.** Go to https://sharkattackfile.net/ and download their dataset on shark attacks.

1. Open the shark attack file using Pandas. It is probably not a csv file, so `read_csv` won't work.
2. Drop any columns that do not contain data.
3. Clean the year variable. Describe the range of values you see. Filter the rows to focus on attacks since 1940. Are attacks increasing, decreasing, or remaining constant over time?
4. Clean the Age variable and make a histogram of the ages of the victims.
5. What proportion of victims are male?
6. Clean the `Type` variable so it only takes three values: Provoked and Unprovoked and Unknown. What proportion of attacks are unprovoked?
7. Clean the `Fatal Y/N` variable so it only takes three values: Y, N, and Unknown.
8. Are sharks more likely to launch unprovoked attacks on men or women? Is the attack more or less likely to be fatal when the attack is provoked or unprovoked? Is it more or less likely to be fatal when the victim is male or female? How do you feel about sharks?
9. What proportion of attacks appear to be by white sharks? (Hint: `str.split()` makes a vector of text values into a list of lists, split by spaces.)

In [54]:
#Question 2
import numpy as np
import pandas as pd
#1.1
url = "https://sharkattackfile.net/spreadsheets/GSAF5.xls"
df = pd.read_excel(url, engine="xlrd")
#print(df.head())

#1.2
df_cleaned = df.dropna()

#1.3
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
#There is a very wide range of years starting from 1845 going all the way to 2025
df_recent = df[df['Year'] >= 1940]
attacks_per_year = df_recent.groupby('Year').size().reset_index(name='Attack Count')
print(attacks_per_year)
#Attacks are increasing over time

#1.4
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df_age_cleaned = df.dropna(subset=['Age'])
print(df_age_cleaned['Age'].describe())
import matplotlib.pyplot as plt
plt.hist(df_age_cleaned['Age'], bins=20, edgecolor='purple', color='pink')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

#1.5
male_count = df[df['Sex'] == 'M'].shape[0]
male_prop = male_count / len(df)
print(male_prop)

#1.6
df['Type'] = df['Type'].astype(str).str.strip().str.lower()
df.loc[df['Type'] == "unprovoked", 'Type'] = "unprovoked"
df.loc[df['Type'] == "provoked", 'Type'] = "provoked"
df.loc[~df['Type'].isin(['unprovoked', 'provoked']), 'Type'] = "unknown"
print(df['Type'].value_counts())

#1.7
df['Fatal Y/N']=df['Fatal Y/N'].astype(str).str.strip().str.upper()
df.loc[df['Fatal Y/N'] == "Y", 'Fatal Y/N'] = "Y"
df.loc[df['Fatal Y/N'] == "N", 'Fatal Y/N'] = "N"
df.loc[~df['Fatal Y/N'].isin(['Y', 'N']), 'Fatal Y/N'] = "Unknown"
print(df['Fatal Y/N'].value_counts())

#1.8
df['Type'] = df['Type'].astype(str).str.strip().str.lower()
unprovoked_attacks = df[df['Type'] == "unprovoked"]
unprovoked_attacks['Sex'] = unprovoked_attacks['Sex'].str.strip().str.upper()
gender_counts = unprovoked_attacks['Sex'].value_counts()
print(gender_counts) #more likely to attack unprovoked at men

unprovoked_attacks.loc[:, 'Sex'] = unprovoked_attacks['Sex'].str.strip().str.upper()
fatality_rates = df.groupby('Type')['Fatal Y/N'].apply(lambda x: (x == 'Y').mean())
print(fatality_rates)
#Attacks seem to be more fatal with unprovoked (.24) than provoked (.03)

df['Fatal Y/N'] = df['Fatal Y/N'].astype(str).str.strip().str.upper()
Fatal_attacks = df[df['Fatal Y/N'] == "Y"].copy()
Fatal_attacks['Sex'] = Fatal_attacks['Sex'].astype(str).str.strip().str.upper()
fatal_gender_counts = Fatal_attacks['Sex'].value_counts()
print(fatal_gender_counts) #more fatal when man is the victim

#I used to be very afraid of sharks but I think they are cool. I like nurse sharks.

#1.9
df['Species '] = df['Species '].astype(str).str.strip().str.lower()
white_shark_count = df[df['Species '].str.contains('white')].shape[0]
total_shark_count = df.shape[0]
white_shark_proportion = white_shark_count / total_shark_count
print(white_shark_proportion) #10.6% are white sharks

0.1066933638443936


**Q3.** Open the "tidy_data.pdf" document in the repo, which is a paper called Tidy Data by Hadley Wickham.

  1. Read the abstract. What is this paper about?
  2. Read the introduction. What is the "tidy data standard" intended to accomplish?
  3. Read the intro to section 2. What does this sentence mean: "Like families, tidy datasets are all alike but every messy dataset is messy in its own way." What does this sentence mean: "For a given dataset, it’s usually easy to figure out what are observations and what are variables, but it is surprisingly difficult to precisely define variables and observations in general."
  4. Read Section 2.2. How does Wickham define values, variables, and observations?
  5. How is "Tidy Data" defined in section 2.3?
  6. Read the intro to Section 3 and Section 3.1. What are the 5 most common problems with messy datasets? Why are the data in Table 4 messy? What is "melting" a dataset?
  7. Why, specifically, is table 11 messy but table 12 tidy and "molten"?
  8. Read Section 6. What is the "chicken-and-egg" problem with focusing on tidy data? What does Wickham hope happens in the future with further work on the subject of data wrangling?

**Q4.** Many important datasets contain a race variable, typically limited to a handful of values often including Black, White, Asian, Latino, and Indigenous. This question looks at data gathering efforts on this variable by the U.S. Federal government.

1. How did the most recent US Census gather data on race?
2. Why do we gather these data? What role do these kinds of data play in politics and society? Why does data quality matter?
3. Please provide a constructive criticism of how the Census was conducted: What was done well? What do you think was missing? How should future large scale surveys be adjusted to best reflect the diversity of the population? Could some of the Census' good practices be adopted more widely to gather richer and more useful data?
4. How did the Census gather data on sex and gender? Please provide a similar constructive criticism of their practices.
5. When it comes to cleaning data, what concerns do you have about protected characteristics like sex, gender, sexual identity, or race? What challenges can you imagine arising when there are missing values? What good or bad practices might people adopt, and why?
6. Suppose someone invented an algorithm to impute values for protected characteristics like race, gender, sex, or sexuality. What kinds of concerns would you have?

1. The 2020 census gathered race data through self-identification. Respondents could select multiple racial categories. The five main racial categories were White, Black or African American, Asian, American Indian or Alaska Narive, and Other Pacfic Islande or Native Hawaiian.

2. This racial data help monitro discimination, see any trends, enforce civil rights laws, and allocate federal resources fairly. This data shapes polcies on public health, education, and economic disparties amoung raced. Poor data can lead to misinformed policies, unfair allocation of resources, and negelect of certian groups.

3. The Census improved representation by allowing multiracial identification and write-in options, but it still relies on predefined categories that may not fully represent racial complexity. Future surveys should include more flexible racial classifications. Expanding outreach in diverse communities and adopting a more inclusive framework would lead to more accurate and more useful data.

4. Sex data were collected by a binary male/female question, without accounting for non-binary or transgender identities. This limits the accuracy of demographic representation and erases gender-diverse individuals from official records. Future Census surveys should include separate gender identity questions to better reflect population diversity.

5. Cleaning protected characteristics like sex, gender, race, and sexual identity requires caution to avoid reinforcing biases. Missing values can introduce bias if nonresponse correlates with marginalized identities, and using default values risks misclassification. Best practices prioritize transparency, self-reported data, and avoiding assumptions about identity.

6. An algorithm imputing race, gender, sex, or sexuality would raise ethical concerns, as these characteristics are very personal and often change a lot. This algorithm could reinforce societal biases, oversimplify identity, and lead to discriminatory outcomes. Any attempt at imputation should be approached cautiously, ensuring equality and harm mitigation.